# Circle fitting example

Suppose that we want to fit a circle in the form:

\begin{equation*}
\phi(x,y) = a x^2 + b_x x + b_y y + a y^2 + c= 0
\end{equation*}

using both the algebraic and the geometric distance.


In [ ]:

def plot_circle(xc,yc,r, color="-r" ):
    a = np.linspace(0,2*np.pi,100)
    px = np.cos(a)*r + xc
    py = np.sin(a)*r + yc
    plt.plot( px, py, color)

In [ ]:
# Let's generate some points
import numpy as np
import matplotlib.pyplot as plt

npts = 100
radius=2

alphas = np.linspace( 0,np.pi/2, npts)
perturb = radius+np.random.randn( npts )* 0.1;
data = np.vstack(  (perturb*np.cos(alphas), perturb*np.sin(alphas) ) ).T



plt.figure( figsize=(8,8))
plt.scatter( data[:,0], data[:,1])
plot_circle(0,0,radius,'-g')
plt.xlim(-3,3)
plt.ylim(-3,3)
plt.grid(True)
plt.legend(["points", "true circle"])

# Algebraic distance

We consider the algebraic distance approximation

\begin{equation*}
\mbox{dist}(p_x,p_y,\phi) = a {p_x}^2 + b_x p_x + b_y p_y + a {p_y}^2 + c
\end{equation*}

and we solve the following least-squares problem:

\begin{equation*}
\mbox{argmin} \sum_{i=1}^N \big( \mbox{dist}(p_{x_i},p_{y_i},\phi) \big)^2
\end{equation*}

with the following normalization constraint on the curve coefficients:

\begin{equation*}
a^2 + b_x^2 + b_y^2 + c^2 = 1
\end{equation*}


In matrix form:

\begin{equation*}
\mbox{argmin} \; \| \mathbf{A} x \|_2^2  \;\; \mbox{s.t.} \|x \|^2 = 1
\end{equation*}

where:

\begin{equation*}
A = \begin{pmatrix}
x_1^2 + y_1^2 & x_1 & y_1 & 1 \\
x_2^2 + y_2^2 & x_2 & y_2 & 1 \\
\vdots & \vdots & \vdots & \vdots \\
x_N^2 + y_N^2 & x_N & y_N & 1
\end{pmatrix} \;, \;\;
x = \begin{pmatrix}
a \\ b_x \\ b_y \\ c
\end{pmatrix}
\end{equation*}

The solution can be found from the right singular vector of $A$ corresponding to the smallest singular value

In [ ]:

x = data[:,0]
y = data[:,1]
A = np.vstack( [x**2 + y**2, x, y, np.ones(npts)]   ).T

_, s, v = np.linalg.svd(A)
print(s)

params = v[3,:]
print(params)


xc = -params[1]/(2*params[0]);
yc = -params[2]/(2*params[0]);
print(xc)
print(yc)
r = np.sqrt(np.abs((params[1]**2 + params[2]**2)/(4*params[0]**2) - params[3]/params[0]));


plt.figure( figsize=(8,8))
plt.scatter( data[:,0], data[:,1])
plt.xlim(-3,3)
plt.ylim(-3,3)
plot_circle(xc,yc,r)
plot_circle(0,0,radius,'-g')
plt.grid(True)
plt.legend(["points", "fitted circle", "true circle"])

# Geometric distance

We parametrize the circle with a center $(x_c, y_c)$ and a radius $r$. The geometric distance to a point $(d_x,d_y)$ is defined as

\begin{equation*}
\mbox{dist}(p_x,p_y,x_c, y_c, r)^2 = \bigg( r- \sqrt{(x_c - p_x)^2 + (y_c -p_y)^2} \bigg)^2
\end{equation*}

resulting in the following ***non-linear*** minimization problem:

\begin{equation*}
\mbox{argmin}_{x_c,y_c,r} \; \sum_{i=1}^N \mbox{dist}_i(x_c,y_c,r)^2
\end{equation*}

where $\mbox{dist}_i(x_c,y_c,r)^2 = \mbox{dist}(p_{x_i},p_{y_i},x_c, y_c, r)^2$.
In vector form we can define a function $K: \mathbb{R}^3 \to \mathbb{R}^N$

\begin{equation*}
K(x_c, y_c, r) = \begin{pmatrix} \mbox{dist}_1(x_c,y_c,r)  \\
\vdots \\
 \mbox{dist}_N(x_c,y_c,r)
\end{pmatrix}
\end{equation*}

and formulate the non-linear least squares problem as:

\begin{equation*}
\mbox{argmin}_{x_c,y_c,r} \; \| K(x_c,y_c,r) \|_2^2
\end{equation*}

To solve the system we'll use the function scipy.optimize.leastsq implementing the Levemberg-Marquardt algorithm. The algorithm is essentially a gradient descent operation using the function Jacobian evaluated at $\mathbf{x}=(x_c,y_c,r)$ to iteratively improve the solution


\begin{equation*}
J(x_c,y_c,r) = \begin{pmatrix}
\frac{\partial \mbox{dist}_1(x_c,y_c,r)}{\partial x_c} & \frac{\partial \mbox{dist}_1(x_c,y_c,r)}{\partial y_c} & \frac{\partial \mbox{dist}_1(x_c,y_c,r)}{\partial r} \\
\vdots & \vdots & \vdots \\
\frac{\partial \mbox{dist}_N(x_c,y_c,r)}{\partial x_c} & \frac{\partial \mbox{dist}_N(x_c,y_c,r)}{\partial y_c} & \frac{\partial \mbox{dist}_N(x_c,y_c,r)}{\partial r} \\
\end{pmatrix}
\end{equation*}

In [ ]:
def dist( x, pts):
    xc = x[0]
    yc = x[1]
    r = x[2]

    return  np.sqrt( (xc-pts[:,0])**2 + (yc-pts[:,1])**2 ) - r


def jac( x, pts):
    xc = x[0]
    yc = x[1]
    r = x[2]

    denom = np.sqrt( (xc-pts[:,0])**2 + (yc-pts[:,1])**2 )

    return np.vstack( [  (xc-pts[:,0])/denom,  (yc-pts[:,1])/denom, -np.ones(pts.shape[0])     ] ).T


pts = np.vstack( [data[:,0], data[:,1]] ).T
x0 = [0,0,1] # Initial guess for xc,yc,radius

import scipy.optimize
out = scipy.optimize.leastsq( dist, x0, args=pts, Dfun=jac, ftol=1e-10 )
params = out[0]
print(params)

plt.figure( figsize=(8,8))
plt.scatter( data[:,0], data[:,1])
plt.xlim(-3,3)
plt.ylim(-3,3)
plot_circle(params[0],params[1],params[2])
plot_circle(0,0,radius,'-g')
plt.grid(True)
plt.legend(["points", "fitted circle", "true circle"])
